# Create Databricks Secret Scope & SQL Server Secrets

Creates a Databricks-backed secret scope and stores two secrets for SQL Server authentication.  
Safe to rerun — skips scope creation if it already exists and always updates the secret values.

In [ ]:
# ------------------------------------------------------------
# 1. Configuration — fill in all values before running
# ------------------------------------------------------------

scope_name    = "sql-server-scope"       # name of the secret scope to create
db_user_key   = "sql-server-user"        # secret key name for the database user
db_pass_key   = "sql-server-password"    # secret key name for the database password

db_user_value = "your_database_user"     # actual database username
db_pass_value = "your_database_password" # actual database password

In [ ]:
# ------------------------------------------------------------
# 2. Resolve workspace URL and token from notebook context
#    No hardcoded credentials needed
# ------------------------------------------------------------
import requests

ctx         = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host        = ctx.apiUrl().get()
token       = ctx.apiToken().get()
auth_header = {"Authorization": f"Bearer {token}"}

print(f"Workspace: {host}")

In [ ]:
# ------------------------------------------------------------
# 3. Create secret scope (skips if it already exists)
# ------------------------------------------------------------

response = requests.post(
    f"{host}/api/2.0/secrets/scopes/create",
    headers=auth_header,
    json={
        "scope": scope_name,
        "initial_manage_principal": "users",
    },
)

if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created.")
elif response.json().get("error_code") == "RESOURCE_ALREADY_EXISTS":
    print(f"Secret scope '{scope_name}' already exists — skipping creation.")
else:
    raise Exception(f"Failed to create scope: {response.text}")

In [ ]:
# ------------------------------------------------------------
# 4. Store secrets (creates or overwrites on every run)
# ------------------------------------------------------------

def put_secret(scope, key, value):
    response = requests.post(
        f"{host}/api/2.0/secrets/put",
        headers=auth_header,
        json={"scope": scope, "key": key, "string_value": value},
    )
    if response.status_code != 200:
        raise Exception(f"Failed to store secret '{key}': {response.text}")
    print(f"Secret '{key}' stored in scope '{scope}'.")

put_secret(scope_name, db_user_key, db_user_value)
put_secret(scope_name, db_pass_key, db_pass_value)

print("\nDone. Reference these secrets in your ingestion notebook as:")
print(f'  dbutils.secrets.get(scope="{scope_name}", key="{db_user_key}")')
print(f'  dbutils.secrets.get(scope="{scope_name}", key="{db_pass_key}")')